# Colour - Checker Detection - Examples: Segmentation

This notebook showcases segmentation retated colour checker detection examples.

<div class="alert alert-info">
The original *.CR2 files were processed with <em>dcraw</em> as follows: <code>dcraw -T -g 2.4 12.92 *.CR2</code> and then resized and converted to *.png with <em>Image Magick</em> as follows: <code>mogrify -resize 50% -format png *.tiff</code>.
</div>

In [1]:
import glob
import numpy as np
import os

import colour
from colour_checker_detection import (
    ROOT_RESOURCES_EXAMPLES,
    detect_colour_checkers_segmentation)

colour.plotting.colour_style()

colour.utilities.describe_environment();

*                                                                             *
*   Interpreter :                                                             *
*       python : 3.10.16 (main, Dec  4 2024, 08:53:37) [GCC 9.4.0]            *
*                                                                             *
*   colour-science.org :                                                      *
*       colour : v0.1.2-248-g07bf5c1                                          *
*       colour-checker-detection : v0.1.2-248-g07bf5c1                        *
*                                                                             *
*   Runtime :                                                                 *
*       imageio : 2.36.1                                                      *
*       matplotlib : 3.10.0                                                   *
*       networkx : 3.4.2                                                      *
*       numpy : 2.2.1                   

## Images

In [2]:
COLOUR_CHECKER_IMAGE_PATHS = glob.glob(
    os.path.join(ROOT_RESOURCES_EXAMPLES, 'colour-checker-detection-dataset/train/images', '*19*.png'))
COLOUR_CHECKER_IMAGES = [
    colour.cctf_decoding(colour.io.read_image(path))
    for path in COLOUR_CHECKER_IMAGE_PATHS
]

for image in COLOUR_CHECKER_IMAGES:
    colour.plotting.plot_image(colour.cctf_encoding(image));

## Detection

In [3]:
SWATCHES = []
for image in COLOUR_CHECKER_IMAGES:
    for colour_checker_data in detect_colour_checkers_segmentation(
        image, additional_data=True):
        
        swatch_colours, swatch_masks, colour_checker_image, _ = (
            colour_checker_data.values)
        SWATCHES.append(swatch_colours)
        
        # Using the additional data to plot the colour checker and masks.
        masks_i = np.zeros(colour_checker_image.shape)
        for i, mask in enumerate(swatch_masks):
            masks_i[mask[0]:mask[1], mask[2]:mask[3], ...] = 1
        
        colour.plotting.plot_image(
            colour.cctf_encoding(
                np.clip(colour_checker_image + masks_i * 0.25, 0, 1)));

## Colour Fitting

In [4]:
D65 = colour.CCS_ILLUMINANTS['CIE 1931 2 Degree Standard Observer']['D65']
REFERENCE_COLOUR_CHECKER = colour.CCS_COLOURCHECKERS[
    'ColorChecker24 - After November 2014']

colour_checker_rows = REFERENCE_COLOUR_CHECKER.rows
colour_checker_columns = REFERENCE_COLOUR_CHECKER.columns

# NOTE: The reference swatches values as produced by the "colour.XYZ_to_RGB"
# definition are linear by default.
# See https://github.com/colour-science/colour-checker-detection/discussions/59
# for more information.
REFERENCE_SWATCHES = colour.XYZ_to_RGB(
        colour.xyY_to_XYZ(list(REFERENCE_COLOUR_CHECKER.data.values())),
        'sRGB', REFERENCE_COLOUR_CHECKER.illuminant)

for i, swatches in enumerate(SWATCHES):
    swatches_xyY = colour.XYZ_to_xyY(colour.RGB_to_XYZ(
        swatches, 'sRGB', D65))

    colour_checker = colour.characterisation.ColourChecker(
        os.path.basename(COLOUR_CHECKER_IMAGE_PATHS[i]),
        dict(zip(REFERENCE_COLOUR_CHECKER.data.keys(), swatches_xyY)),
        D65, colour_checker_rows, colour_checker_columns)
    
    colour.plotting.plot_multi_colour_checkers(
        [REFERENCE_COLOUR_CHECKER, colour_checker])
    
    swatches_f = colour.colour_correction(swatches, swatches, REFERENCE_SWATCHES)
    swatches_f_xyY = colour.XYZ_to_xyY(colour.RGB_to_XYZ(
        swatches_f, 'sRGB', D65))
    colour_checker = colour.characterisation.ColourChecker(
        '{0} - CC'.format(os.path.basename(COLOUR_CHECKER_IMAGE_PATHS[i])),
        dict(zip(REFERENCE_COLOUR_CHECKER.data.keys(), swatches_f_xyY)),
        D65, colour_checker_rows, colour_checker_columns)
    
    colour.plotting.plot_multi_colour_checkers(
        [REFERENCE_COLOUR_CHECKER, colour_checker])

    colour.plotting.plot_image(colour.cctf_encoding(
        colour.colour_correction(
            COLOUR_CHECKER_IMAGES[i], swatches, REFERENCE_SWATCHES)));

## Additional Data Plotting

In [5]:
for image in COLOUR_CHECKER_IMAGES:
    for colour_checker_data in detect_colour_checkers_segmentation(
            image, show=True):
        pass

# Failures

The current segmentation process is prone to fail if the image is not normal to the optical path, in such cases, it is recommended to use the machine learning inference approach:

<div class="alert alert-info">
The image path can also be passed directly to the detection definition and that it is possible to control the image decoding directly.
</div>

In [6]:
COLOUR_CHECKER_IMAGE_PATHS = glob.glob(
    os.path.join(ROOT_RESOURCES_EXAMPLES, "colour-checker-detection-dataset/train/images", "*25*.png")
)

for path in COLOUR_CHECKER_IMAGE_PATHS:
    for colour_checker_data in detect_colour_checkers_segmentation(
        path, apply_cctf_decoding=True, show=True
    ):
        pass